# Portfolio half-hourly forecast — one model for all meters

Forecast each meter's half-hourly kWh one day ahead (48 periods) with a single Ridge
model trained across the whole portfolio, using lags, a rolling level and calendar features.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 120)

df = pd.read_csv("../data/meter_halfhourly_2023.csv.gz")
meters = pd.read_csv("../data/meters.csv")
df["utc"] = pd.to_datetime(df["settlement_date"]) + pd.to_timedelta((df["settlement_period"] - 1) * 30, unit="min")
df = df.sort_values(["meter_id", "utc"]).reset_index(drop=True)
df.shape

(349439, 5)

## Features

Lags of one and seven days, a rolling daily level aligned to the forecast day, calendar
features, and a per-meter z-score so that big and small meters are on a comparable scale.

In [2]:
df["lag48"] = df["kwh"].shift(48)
df["lag336"] = df["kwh"].shift(336)
df["roll48"] = df["kwh"].rolling(48).mean().shift(-48)
df["target"] = df["kwh"].shift(-48)

df["hour"] = df["utc"].dt.hour
df["dow"] = df["utc"].dt.dayofweek
df["is_weekend"] = (df["dow"] >= 5).astype(int)
df[["meter_id", "utc", "kwh", "lag48", "roll48", "target"]].iloc[400:405]

,meter_id,utc,kwh,lag48,roll48,target
400,M100000,2023-01-09 07:30:00,0.448,0.577,1.784333,0.682
401,M100000,2023-01-09 08:00:00,3.073,1.305,1.821292,4.847
402,M100000,2023-01-09 08:30:00,2.635,3.065,1.839021,3.486
403,M100000,2023-01-09 09:00:00,3.863,3.052,1.804792,2.220
404,M100000,2023-01-09 09:30:00,4.715,3.705,1.764437,2.778


In [3]:
stats = df.groupby("meter_id")["kwh"].agg(["mean", "std"])
df = df.merge(stats, left_on="meter_id", right_index=True)
for c in ["kwh", "lag48", "lag336", "roll48", "target"]:
    df[c + "_z"] = (df[c] - df["mean"]) / df["std"]

dummies = pd.get_dummies(df["meter_id"], prefix="m", drop_first=True).astype(float)
hour_d = pd.get_dummies(df["hour"], prefix="h").astype(float)
X = pd.concat([df[["kwh_z", "lag48_z", "lag336_z", "roll48_z", "is_weekend"]], hour_d, dummies], axis=1)
y = df["target_z"]
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]
X.shape

(349055, 48)

## Model

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
model = Ridge(alpha=1.0).fit(X_train, y_train)
pred_z = model.predict(X_test)
print("R2 (z-scored):", round(r2_score(y_test, pred_z), 4))

R2 (z-scored): 0.5356


In [5]:
coef = pd.Series(model.coef_, index=X.columns)
coef[[c for c in coef.index if not c.startswith(("m_", "h_"))]].round(4)

kwh_z         0.2812
lag48_z       0.1767
lag336_z      0.0292
roll48_z      0.5950
is_weekend    0.0225
dtype: float64

Convert back to kWh for reporting.

In [6]:
res = df.loc[X_test.index, ["meter_id", "utc", "target", "mean", "std"]].copy()
res["pred"] = pred_z * res["std"] + res["mean"]
r2_kwh = r2_score(res["target"], res["pred"])
rmse_kwh = np.sqrt(mean_squared_error(res["target"], res["pred"]))
print("R2 (kWh):", round(r2_kwh, 4), "| RMSE (kWh):", round(rmse_kwh, 4))

R2 (kWh): 0.7907 | RMSE (kWh): 6.5548


## Per-meter effects

Meter dummies capture each meter's level relative to the baseline. M100000 has no dummy
because it is the reference, so its effect is zero.

In [7]:
meter_eff = coef[[c for c in coef.index if c.startswith("m_")]].sort_values()
meter_eff.round(4)

m_M100001   -0.0123
m_M100016   -0.0069
m_M100011   -0.0009
m_M100003    0.0030
m_M100018    0.0031
m_M100017    0.0038
m_M100015    0.0040
m_M100006    0.0057
m_M100010    0.0059
m_M100004    0.0063
m_M100007    0.0077
m_M100013    0.0090
m_M100012    0.0092
m_M100019    0.0092
m_M100002    0.0099
m_M100005    0.0110
m_M100008    0.0111
m_M100009    0.0300
m_M100014    0.0391
dtype: float64

## Fit by customer type

In [8]:
res = res.merge(meters[["meter_id", "customer_type"]], on="meter_id")
res.groupby("customer_type")[["target", "pred"]].apply(lambda g: pd.Series({
    "n": len(g), "r2": r2_score(g["target"], g["pred"]),
    "rmse": np.sqrt(mean_squared_error(g["target"], g["pred"]))})).round(4)

,n,r2,rmse
customer_type,,,
residential,59458.0,0.7911,7.0903
sme,10353.0,0.5608,1.0022


## Results

- One Ridge model across 20 meters reaches R² ≈ 0.79 in kWh on held-out rows; the rolling
  level is the strongest feature.
- SME meters are forecast almost perfectly; residential meters are noisier but fine.
- Meter dummies pick up the level differences. Ready to run daily for the portfolio.

In [9]:
pd.Series({"r2_test_kwh": round(r2_kwh, 4),
           "rmse_test_kwh": round(rmse_kwh, 4),
           "r2_test_z": round(r2_score(y_test, pred_z), 4),
           "n_train": len(X_train), "n_test": len(X_test)})

r2_test_kwh           0.7907
rmse_test_kwh         6.5548
r2_test_z             0.5356
n_train          279244.0000
n_test            69811.0000
dtype: float64